# 🫁 LUNGRAY DATA PREPARATION: COVIDx CXR-4 MEDICAL LETTERBOXING ENGINE (512x512)
## Chuẩn Hóa 16,400+ Ảnh COVIDx CXR-4 Về Chuẩn $512 \times 512$ Bảo Toàn Tỷ Lệ Giải Phẫu Lồng Ngực

---

### 🎯 Mục tiêu kỹ thuật:
1. **Bảo tồn tỷ lệ hình học giải phẫu (Aspect Ratio Preservation)**: Sử dụng phương pháp **Medical Letterboxing** (`LongestMaxSize(512)` kết hợp viền đen đối xứng) để các cơ quan lồng ngực (bóng tim, xương sườn, cơ hoành) không bị kéo giãn hoặc biến dạng méo mó.
2. **0 Byte ổ cứng local**: Thực thi 100% trên Kaggle Cloud bằng 4 vCPUs đa luồng trong ~3-5 phút.
3. **Tạo Dataset `lungray-covidx-512`**: Xuất cấu trúc thư mục sạch gồm `train/`, `test/` kèm file chỉ mục chính thức `train.txt` và `test.txt` (~2 GB).

### 📌 Hướng dẫn thêm Data trên Kaggle:
- Bấm **+ Add Data** ở góc phải màn hình Notebook.
- Tìm kiếm và thêm dataset: **`andyczhao/covidx-cxr2`** (Kaggle mount ngay trong 0 giây tại `/kaggle/input/covidx-cxr2`).

In [1]:
# CELL 1: Kiểm tra môi trường Kaggle & Định vị COVIDx CXR-4
import os
import sys
import shutil
import time
import glob
import cv2
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

cpu_cores = os.cpu_count() or 4
potential_roots = [
    '/kaggle/input/datasets/andyczhao/covidx-cxr2',
    '/kaggle/input/andyczhao/covidx-cxr2',
    '/kaggle/input/datasets/covidx-cxr2',
    '/kaggle/input/covidx-cxr2',
    '/kaggle/input/covidx-cxr-4',
    'dataset/covidx-cxr2'
]

covid_root = None
train_txt = None
test_txt = None

for root in potential_roots:
    if os.path.exists(root):
        c_train = os.path.join(root, 'train.txt')
        c_test = os.path.join(root, 'test.txt')
        if os.path.exists(c_train) and os.path.exists(c_test):
            covid_root = root
            train_txt = c_train
            test_txt = c_test
            break

if covid_root is None:
    raise FileNotFoundError("[LỖI] Không tìm thấy dataset COVIDx CXR-4! Hãy kiểm tra đường dẫn dataset.")

print(f'[OK] Thư mục gốc COVIDx: {covid_root}')
print(f'[*] train.txt: {train_txt}')
print(f'[*] test.txt:  {test_txt}')


FileNotFoundError: [LỖI] Không tìm thấy dataset COVIDx CXR-4! Hãy đảm bảo đã Add Data 'andyczhao/covidx-cxr2'.

In [ ]:
# CELL 2: Đọc danh sách file ảnh và kiểm toán số lượng mẫu
def parse_covidx_txt(txt_path):
    records = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                records.append({
                    'patient_id': parts[0],
                    'filename': parts[1],
                    'label': parts[2],
                    'data_source': parts[3] if len(parts) >= 4 else 'unknown'
                })
    return records

train_records = parse_covidx_txt(train_txt)
test_records = parse_covidx_txt(test_txt)

print(f'[OK] Tập Train: {len(train_records):,} mẫu')
print(f'[OK] Tập Test:  {len(test_records):,} mẫu')
print(f'[*] Tổng cộng:  {len(train_records) + len(test_records):,} ảnh cần xử lý.')

In [ ]:
# CELL 3: Hàm Medical Letterboxing chuẩn Y khoa
TARGET_SIZE = 512
PNG_COMPRESSION = 4

def letterbox_image(img, target_size=512):
    h, w = img.shape[:2]
    if h == 0 or w == 0:
        return img
    scale = target_size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    
    # Resize cạnh lớn nhất bằng INTER_AREA
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # Tính toán padding đối xứng viền đen
    pad_top = (target_size - new_h) // 2
    pad_bottom = target_size - new_h - pad_top
    pad_left = (target_size - new_w) // 2
    pad_right = target_size - new_w - pad_left
    
    padded = cv2.copyMakeBorder(
        resized,
        pad_top, pad_bottom, pad_left, pad_right,
        borderType=cv2.BORDER_CONSTANT,
        value=0
    )
    return padded

def process_covidx_job(job):
    src_path, dst_path = job
    if os.path.exists(dst_path):
        return True
    try:
        img = cv2.imread(src_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            return False
        letterboxed = letterbox_image(img, TARGET_SIZE)
        cv2.imwrite(dst_path, letterboxed, [cv2.IMWRITE_PNG_COMPRESSION, PNG_COMPRESSION])
        return True
    except Exception:
        return False

In [ ]:
# CELL 4: Chuẩn bị danh sách đường dẫn và thư mục output
output_root = '/kaggle/working/covidx_512' if os.path.exists('/kaggle') else 'data/processed/covidx_512'
out_train_dir = os.path.join(output_root, 'train')
out_test_dir = os.path.join(output_root, 'test')
os.makedirs(out_train_dir, exist_ok=True)
os.makedirs(out_test_dir, exist_ok=True)

jobs = []
for split_name, records, out_dir in [('train', train_records, out_train_dir), ('test', test_records, out_test_dir)]:
    for r in records:
        fname = r['filename']
        cand1 = os.path.join(covid_root, split_name, fname)
        cand2 = os.path.join(covid_root, fname)
        src = cand1 if os.path.exists(cand1) else cand2
        dst = os.path.join(out_dir, fname)
        jobs.append((src, dst))

print(f'[*] Đã tạo danh sách {len(jobs):,} tác vụ Letterbox cho cả train và test.')

In [ ]:
# CELL 5: Thực thi Letterboxing song song với ProcessPoolExecutor
start_time = time.time()
success_count = 0

with ProcessPoolExecutor(max_workers=cpu_cores) as executor:
    for res in tqdm(executor.map(process_covidx_job, jobs, chunksize=100), total=len(jobs), desc='Letterboxing COVIDx'):
        if res:
            success_count += 1

elapsed = time.time() - start_time
print(f'\n[OK] Hoàn tất sau {elapsed/60:.2f} phút ({len(jobs)/max(elapsed, 1):.1f} ảnh/giây)!')
print(f'[OK] Đã ghi thành công: {success_count:,} / {len(jobs):,} ảnh.')

In [ ]:
# CELL 6: Trực quan hóa đối chứng Before vs After (Bảo toàn tỷ lệ giải phẫu)
sample_jobs = jobs[:3]
fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for idx, (src, dst) in enumerate(sample_jobs):
    img_raw = cv2.imread(src)
    img_out = cv2.imread(dst)
    
    axes[idx, 0].imshow(cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB))
    axes[idx, 0].set_title(f'GỐC: {img_raw.shape[1]}x{img_raw.shape[0]}', fontsize=11)
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB))
    axes[idx, 1].set_title(f'LETTERBOX: {img_out.shape[1]}x{img_out.shape[0]} (512x512)', fontsize=11)
    axes[idx, 1].axis('off')

plt.suptitle('Đối chứng Mẫu Trước vs Sau Khi Áp Dụng Medical Letterboxing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Sao chép train.txt và test.txt sang thư mục output
shutil.copy2(train_txt, os.path.join(output_root, 'train.txt'))
shutil.copy2(test_txt, os.path.join(output_root, 'test.txt'))
print('[OK] Đã sao chép train.txt và test.txt vào thư mục output.')

In [ ]:
# CELL 8: Đo tổng dung lượng đầu ra
total_bytes = sum(os.path.getsize(os.path.join(r, f)) for r, _, files in os.walk(output_root) for f in files)
total_mb = total_bytes / (1024**2)

print('=' * 60)
print('🫁 TỔNG KẾT BỘ DỮ LIỆU COVIDx 512x512 MỚI:')
print(f'- Thư mục output: {output_root}')
print(f'- Dung lượng tổng: {total_mb:.2f} MB (~{total_mb / 1024:.2f} GB)')
print('=' * 60)
print('[HOÀN TOÀN AN TOÀN] Dung lượng cực kỳ nhẹ, tối ưu tuyệt đối cho quá trình nạp ảnh!')

---

## 🚀 HƯỚNG DẪN 1-CLICK TẠO KAGGLE DATASET `lungray-covidx-512`

Sau khi notebook chạy xong các cell trên:
1. Bấm **Save Version** $\to$ Chọn **Save & Run All (Commit)** $\to$ Bấm **Save**.
2. Sau khi commit hoàn tất, vào tab **Output** ở thanh bên phải.
3. Tại mục `covidx_512`, bấm dấu **3 chấm (...)** $\to$ Chọn **Create New Dataset**.
4. Đặt tiêu đề dataset: **`lungray-covidx-512`** và bấm **Create**.
5. Dataset `lungray-covidx-512` đã sẵn sàng để mount vào notebook huấn luyện chung!